In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./datasets/ecomm.csv", names=["category", "description"], header=None)
df

,category,description
0,Household,Paper Plane Design Framed Wall Hanging Motivat...
1,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,Household,SAF 'UV Textured Modern Art Print Framed' Pain...
3,Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
4,Household,Incredible Gifts India Wooden Happy Birthday U...
...,...,...
50420,Electronics,Strontium MicroSD Class 10 8GB Memory Card (Bl...
50421,Electronics,CrossBeats Wave Waterproof Bluetooth Wireless ...
50422,Electronics,Karbonn Titanium Wind W4 (White) Karbonn Titan...
50423,Electronics,"Samsung Guru FM Plus (SM-B110E/D, Black) Colou..."


In [3]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27802 entries, 0 to 50410
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   category     27802 non-null  object
 1   description  27802 non-null  object
dtypes: object(2)
memory usage: 651.6+ KB


In [4]:
df["category"].value_counts()


category
Household                 10564
Books                      6256
Clothing & Accessories     5674
Electronics                5308
Name: count, dtype: int64

In [5]:
df["category"] = df["category"].map(
    {
        "Household": "household",
        "Books": "books",
        "Clothing & Accessories": "clothing_accessories",
        "Electronics": "electronics",
    }
)

In [6]:
df["description"][0]

'Paper Plane Design Framed Wall Hanging Motivational Office Decor Art Prints (8.7 X 8.7 inch) - Set of 4 Painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch. This painting is ready to hang, you would be proud to possess this unique painting that is a niche apart. We use only the most modern and efficient printing technology on our prints, with only the and inks and precision epson, roland and hp printers. This innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime. We print solely with top-notch 100% inks, to achieve brilliant and true colours. Due to their high level of uv resistance, our prints retain their beautiful colours for many years. Add colour and style to your living space with this digitally printed painting. Some are for pleasure and some for eternal blis

In [7]:
import re


def preprocess(text):
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"[ \n]+", " ", text)

    return text.lower().strip()


In [8]:
preprocess(df["description"][0])

'paper plane design framed wall hanging motivational office decor art prints 8 7 x 8 7 inch set of 4 painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it this is an special series of paintings which makes your wall very beautiful and gives a royal touch this painting is ready to hang you would be proud to possess this unique painting that is a niche apart we use only the most modern and efficient printing technology on our prints with only the and inks and precision epson roland and hp printers this innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime we print solely with top notch 100 inks to achieve brilliant and true colours due to their high level of uv resistance our prints retain their beautiful colours for many years add colour and style to your living space with this digitally printed painting some are for pleasure and some for eternal bliss so bring home th

In [9]:
df["description"] = df["description"].apply(preprocess)
df = df[df["description"].str.len() > 0]

df

,category,description
0,household,paper plane design framed wall hanging motivat...
1,household,saf floral framed painting wood 30 inch x 10 i...
2,household,saf uv textured modern art print framed painti...
3,household,saf flower print framed painting synthetic 13 ...
4,household,incredible gifts india wooden happy birthday u...
...,...,...
50402,electronics,micromax bharat 5 plus zero impact on visual d...
50403,electronics,microsoft lumia 550 8gb 4g black microsoft lum...
50407,electronics,microsoft lumia 535 black 8gb colour black pro...
50408,electronics,karbonn titanium wind w4 white karbonn titaniu...


In [10]:
df["labeled_description"] = (
    "__label__" + df["category"].astype(str) + " " + df["description"]
)

df["labeled_description"]


0        __label__household paper plane design framed w...
1        __label__household saf floral framed painting ...
2        __label__household saf uv textured modern art ...
3        __label__household saf flower print framed pai...
4        __label__household incredible gifts india wood...
                               ...                        
50402    __label__electronics micromax bharat 5 plus ze...
50403    __label__electronics microsoft lumia 550 8gb 4...
50407    __label__electronics microsoft lumia 535 black...
50408    __label__electronics karbonn titanium wind w4 ...
50410    __label__electronics nokia lumia 530 dual sim ...
Name: labeled_description, Length: 27802, dtype: object

In [11]:
df["category"].value_counts()


category
household               10564
books                    6256
clothing_accessories     5674
electronics              5308
Name: count, dtype: int64

In [12]:
labels = df["category"].unique()
labels


array(['household', 'books', 'clothing_accessories', 'electronics'],
      dtype=object)

In [13]:
print("Before", df.shape)
groups = [df[df["category"] == label] for label in labels]
n = len(max(groups, key=len))

sampled = [group.sample(n, replace=True, random_state=42) for group in groups]
df = pd.concat(sampled, ignore_index=True, axis=0)
print("After", df.shape)

Before (27802, 3)
After (42256, 3)


In [14]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df["labeled_description"], test_size=0.2, random_state=42
)


In [15]:
train.to_csv("./datasets/ecomm.train", header=False, index=False)
test.to_csv("./datasets/ecomm.test", header=False, index=False)

In [17]:
import fasttext

model = fasttext.train_supervised("./datasets/ecomm.train")


Read 3M words
Number of words:  67518
Number of labels: 4
Progress: 100.0% words/sec/thread: 5395670 lr:  0.000000 avg.loss:  0.224090 ETA:   0h 0m 0s


In [18]:
model.test("./datasets/ecomm.test")

(8452, 0.9700662565073356, 0.9700662565073356)